# ASR demo

In [1]:
!git clone -q https://github.com/annapo28/asr.git
%cd asr
!pip install -q -r requirements.txt

/content/asr
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.9/859.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.3/226.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 27.9 MB/s eta 0:0

In [2]:
!python scripts/download_checkpoint.py

Done: 53.5 MB


## Inference

In [3]:
DATASET_LINK = "https://drive.google.com/file/d/1oeyI39sTnrWi39WH4_1O5X6P9ZpoNfLI/view?usp=sharing"

In [4]:
import os
from pathlib import Path

!rm -rf /content/user_data && mkdir -p /content/user_data
if "drive/folders" in DATASET_LINK:
    !gdown -q --folder "$DATASET_LINK" -O /content/user_data
else:
    !gdown -q --fuzzy "$DATASET_LINK" -O /content/user_data/dataset.zip
    !unzip -q /content/user_data/dataset.zip -d /content/user_data && rm /content/user_data/dataset.zip

DATA_DIR = next(p.parent for p in Path("/content/user_data").rglob("audio") if p.is_dir())
print("Dataset directory:", DATA_DIR)
print("Utterances:", len(list((DATA_DIR / "audio").iterdir())))

Dataset directory: /content/user_data/my_data
Utterances: 10


In [5]:
!python inference.py datasets.test.data_dir="$DATA_DIR" inferencer.save_path=/content/predictions/user

Conformer(
  (subsampling): ConvSubsampling(
    (conv): Sequential(
      (0): Conv2d(1, 144, kernel_size=(3, 3), stride=(2, 2))
      (1): ReLU()
      (2): Conv2d(144, 144, kernel_size=(3, 3), stride=(2, 2))
      (3): ReLU()
    )
    (linear): Linear(in_features=2736, out_features=144, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (blocks): ModuleList(
    (0-7): 8 x ConformerBlock(
      (ff1): FeedForwardModule(
        (net): Sequential(
          (0): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=144, out_features=576, bias=True)
          (2): SiLU()
          (3): Dropout(p=0.1, inplace=False)
          (4): Linear(in_features=576, out_features=144, bias=True)
          (5): Dropout(p=0.1, inplace=False)
        )
      )
      (mhsa): MultiHeadSelfAttentionModule(
        (norm): LayerNorm((144,), eps=1e-05, elementwise_affine=True)
        (attention): RelPositionMultiHeadAttention(
          (q): Linear(in_features

Predictions are in `/content/predictions/user/test`.

In [6]:
if (DATA_DIR / "transcriptions").exists():
    !python calc_metrics.py --gt_dir "$DATA_DIR/transcriptions" --pred_dir /content/predictions/user/test
else:
    print("No transcriptions directory, metrics are not computed.")

Utterances: 10
WER: 23.25
CER: 6.58
